# STM32N6 NPU Baseline Performance Analysis
This notebook analyzes the data collected in the `performance_baseline_results.csv` file to extract formal insights about the hardware capabilities of the NPU. This data is structured to be included in an academic thesis.

## Objective
Understand how Inference Time scales based on:
1. Number of Filters (Perfect Linearity)
2. Kernel Size (1x1, 3x3, 5x5, 7x7) and Physical Limits
3. Hardware Operator Fusion (Conv2D + MaxPool)
4. Architectural Optimization (Depthwise Convolution)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Plot styling configuration for academic report
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)

# Load Data
df = pd.read_csv('performance_baseline_results_devcloudaicore4.0.csv')

# Data cleaning: force numeric columns (FAIL strings will become NaN)
df['Inference Time (ms)'] = pd.to_numeric(df['Inference Time (ms)'], errors='coerce')
df['MACC'] = pd.to_numeric(df['MACC'], errors='coerce')
df

---
## 1. Linearity with respect to Number of Filters (3x3 Kernel)
We tested a 3x3 convolution on a 64x64x3 input image, varying the number of output filters (16, 32, 64).

In [ ]:
df_filters = df[df['Model Name'].str.contains('3x3') & 
                ~df['Model Name'].str.contains('pool') & 
                ~df['Model Name'].str.contains('depthwise') & 
                ~df['Model Name'].str.contains('16f') == False].copy()

# Dynamically extract only the three pure Conv2D 3x3 models
df_filters = df[df['Model Name'].isin([
    'baseline_conv2d_16f_3x3_int8', 
    'baseline_conv2d_3x3_int8', 
    'baseline_conv2d_64f_3x3_int8'
])].sort_values('Filters')

fig, ax1 = plt.subplots(figsize=(9, 5))

color = 'tab:red'
ax1.set_xlabel('Number of Filters', fontweight='bold')
ax1.set_ylabel('Inference Time (ms)', color=color, fontweight='bold')
ax1.plot(df_filters['Filters'], df_filters['Inference Time (ms)'], marker='o', color=color, linewidth=2, markersize=8)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(df_filters['Filters'])

ax2 = ax1.twinx()  
color = 'tab:blue'
ax2.set_ylabel('MACC', color=color, fontweight='bold')
ax2.plot(df_filters['Filters'], df_filters['MACC'], marker='s', linestyle='--', color=color, linewidth=2, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Scalability: Inference Time and MACC vs Number of Filters (3x3 Kernel)')
fig.tight_layout()  
plt.show()

> **Technical Insight:** The hardware exhibits almost **perfect linear scaling**. Doubling the filters (16 -> 32 -> 64) doubles the MACCs, and the inference time grows proportionally in a mathematical and symmetrical way (0.086 -> 0.155 -> 0.291 ms). This makes the performance of the STM32N6 NPU highly predictable.

---
## 2. Impact of Kernel Size and Physical Limits (32 Filters)
We compare 1x1, 3x3, 5x5, and 7x7 kernels while keeping the filter count fixed at 32. 

*(Note: The 7x7 model resulted in a hardware TIMEOUT error due to RAM saturation)*

In [ ]:
df_kernels = df[(df['Filters'] == 32) & 
                ~df['Model Name'].str.contains('depthwise')].copy()

# Use Kernel Size column for correct ordering
kernel_order = ['1x1', '3x3', '5x5', '7x7']
df_kernels['Kernel Size'] = pd.Categorical(df_kernels['Kernel Size'], categories=kernel_order, ordered=True)
df_kernels = df_kernels.sort_values('Kernel Size')

fig, ax = plt.subplots(figsize=(9, 5))
bars = sns.barplot(x='Kernel Size', y='Inference Time (ms)', data=df_kernels, palette='viridis', ax=ax)

for i, bar in enumerate(bars.patches):
    val = bar.get_height()
    if pd.notna(val) and val > 0:
        ax.annotate(f'{val:.3f}', 
                    (bar.get_x() + bar.get_width() / 2., val), 
                    ha='center', va='center', 
                    xytext=(0, 9), textcoords='offset points')

# Add annotation for the failed 7x7 kernel (which is NaN in the plot)
ax.text(3, 0.1, 'TIMEOUT\n(Out of RAM\n1.5MB)', color='red', ha='center', va='center', fontweight='bold', fontsize=12)
plt.title('Inference Time vs Kernel Size (Fixed 32 Filters)')
plt.ylim(0, 0.35)
plt.show()

> **Technical Insight:**
> - **1x1 Kernel**: Required the automatic allocation of 2 hardware epochs. Consequently, it yields a paradoxically higher inference time (0.288 ms) than a 3x3 or 5x5 kernel, as the data transfer overhead on the DDR bus outweighs the actual computation.
> - **3x3 and 5x5 Kernels**: Executed in 1 single optimized epoch, allowing the accelerator to fully leverage spatial parallelism.
> - **7x7 Kernel (Bottleneck)**: Exceeds the NPU's local physical memory (allocating ~1.5MB). This forces massive reliance on system memory, causing bus congestion and a hardware timeout even when the Epoch Controller (EC) is enabled. This demonstrates the critical importance of restricting kernel sizes in silicon for Edge AI.

---
## 3. Hardware Operator Fusion (Conv2D vs Conv2D + MaxPool)
We verify whether the STM32N6 NPU can fuse a subsequent non-convolutional operation (MaxPooling) within the same logical block without a time penalty.

In [ ]:
df_pool = df[df['Model Name'].isin(['baseline_conv2d_16f_3x3_int8', 'baseline_conv2d_pool_int8'])]

plt.figure(figsize=(7, 4))
ax = sns.barplot(x='Layer Type', y='Inference Time (ms)', data=df_pool, palette='Set2')

for i in ax.containers:
    ax.bar_label(i, fmt='%.3f ms', padding=3, fontweight='bold')

plt.title('Impact of MaxPooling (Hardware Operator Fusion)')
plt.ylim(0, 0.12)
plt.show()

> **Technical Insight:** Adding a 2x2 Max Pooling layer after a convolution incurs virtually zero timing cost (+0.002 ms). The NPU successfully fuses the mathematical operations: pooling is applied "on-the-fly" in the registers or L1 memory before writing back to external memory, drastically cutting down I/O time (Nodes=1).

---
## 4. Architectural Optimization: Standard vs Depthwise (3x3)
We compare a standard 3x3 convolution against a 3x3 Depthwise Convolution. Depthwise convolutions are the architectural cornerstone of lightweight models like MobileNet.

In [ ]:
df_depth = df[df['Model Name'].isin(['baseline_conv2d_16f_3x3_int8', 'baseline_depthwise_3x3_int8'])]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(x='Layer Type', y='Inference Time (ms)', data=df_depth, palette='magma', ax=axes[0])
axes[0].set_title('Inference Time: Conv2D(16f) vs Depthwise')
for i in axes[0].containers:
    axes[0].bar_label(i, fmt='%.3f ms', padding=3, fontweight='bold')

sns.barplot(x='Layer Type', y='MACC', data=df_depth, palette='magma', ax=axes[1])
axes[1].set_title('Required Computations (MACC)')
for i in axes[1].containers:
    axes[1].bar_label(i, padding=3, fontweight='bold')

plt.tight_layout()
plt.show()

> **Technical Insight:** Replacing a standard spatial convolution with a *Depthwise* convolution allows the model to process channels independently. This leads to a dramatic drop in required MACCs, and the Inference Time plummets to just `0.031 ms`. 
> 
> This provides strong empirical validation of the extreme efficiency of architectures like *MobileNet* on Edge NPU processors such as the STM32N6.